# Chapter 27 — PyMongo CRUD, query safety, indexes, and MongoDB lifecycle

**Book role:** Databases  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Connect with MongoClient from explicit configuration.
- Use insert, find, update, and delete operations carefully.
- Apply projections, limits, indexes, and timeouts.
- Close client resources and keep live operations outside GUI callbacks.

## 1. MongoClient and deployment configuration

`MongoClient` manages connections to a MongoDB deployment and is generally designed to be reused rather than created for every button click. Use an environment variable for the URI, configure server-selection/connect/socket timeouts appropriate to the application, and do not print the URI because it may contain credentials. The URI’s TLS, authentication, replica-set, and database options depend on deployment configuration.

A connection being constructible does not always mean that a server selection/query will succeed at that instant. Handle reachability failures at the operation boundary. A GUI should distinguish a missing optional URI, authentication failure, DNS/network failure, and a query/schema error where possible, while avoiding overly detailed messages that expose private infrastructure.

### Worked example 1: Create a Mongo client only when configured

Avoid creating a client unconditionally at notebook import. In production, keep it in the application’s repository/service lifetime and close it at shutdown.

In [ ]:
import os
from pymongo import MongoClient

MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    print("MONGO_URI is not set; MongoDB actions remain unavailable.")
else:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000, connectTimeoutMS=5000)
    # Run only in a configured environment:
    # client.admin.command("ping")
    # db = client[os.getenv("MONGO_DATABASE", "analytics")]
    # orders = db["orders"]
    # Close the client on application shutdown: client.close()
    print("MongoDB client configured; perform an explicit ping before enabling actions.")

## 2. CRUD and safe update patterns

`insert_one()` and `insert_many()` create documents; `find_one()` and `find()` retrieve them; `update_one()` can update matching documents using operators such as `$set` and `$inc`; `delete_one()` removes a matching document. Use stable filters—often a unique `_id` or a validated business key—and inspect `matched_count`, `modified_count`, `inserted_id`, or `deleted_count` before reporting the outcome.

Destructive actions should require explicit scope. A blank search filter must never accidentally become `{}` for a delete operation. Use dedicated repository methods with fixed criteria, require a confirmation that states the record or count, and consider soft-delete/audit patterns when business and retention requirements call for them. Avoid exposing arbitrary update documents to a GUI text field.

### Worked example 2: Read with projection, sort, and limit

This function deliberately limits preview size and projects columns. It assumes `collection` is already a configured PyMongo collection and dates are stored in a sortable date type.

In [ ]:
def fetch_order_preview(collection, region: str, limit: int = 200) -> list[dict]:
    if not 1 <= limit <= 2000:
        raise ValueError("limit must be between 1 and 2000")
    criteria = {} if region == "All" else {"region": region}
    projection = {"_id": 1, "order_id": 1, "order_date": 1, "region": 1, "status": 1}
    cursor = collection.find(criteria, projection).sort("order_date", -1).limit(limit)
    try:
        return list(cursor)
    finally:
        cursor.close()

## 3. Projection, limiting, and index strategy

Use a projection to return the fields required for a table, not entire documents with nested private details. Sort by a deterministic field and use a limit for preview screens. For large results, paginate or iterate a cursor in bounded batches rather than eagerly loading every document into memory. Close cursors when abandoning a large query early, and normalize the output in a data adapter.

Create indexes based on real filter/sort usage; examine execution plans and data size. Compound index field order matters. Adding an index has a cost for every write and consumes storage, so indexes should be evaluated, not collected without a workload model. If an app supports region, status, and date filters, select indexes that match the most common query patterns and validate them with `explain`.

### Worked example 3: Update a specific record and verify the outcome

In real data, ensure `order_id` is uniquely indexed or filter by `_id`. A modified count of zero can mean the value was already the requested value, not necessarily that the operation failed.

In [ ]:
def set_order_status(collection, order_id: str, new_status: str) -> int:
    allowed_statuses = {"Pending", "Paid", "Cancelled", "Returned"}
    if new_status not in allowed_statuses:
        raise ValueError("Unsupported order status")
    result = collection.update_one(
        {"order_id": order_id},
        {"$set": {"status": new_status}},
    )
    if result.matched_count == 0:
        return 0
    return int(result.modified_count)

## Guided lab: Lab: CRUD adapter with safety

Create a PyMongo repository supporting list preview, get-by-ID, safe update status, and a separately guarded delete action. Use a projection, result limits, a fixed status allow-list, appropriate index definitions, a configured timeout, and explicit client cleanup. Test “record not found,” “same value,” invalid status, and blank-ID cases without relying on a production collection.

## Review questions and practice

1. Why should a long-lived application reuse MongoClient?
2. Why is an empty delete filter dangerous?
3. What is the distinction between matched and modified counts?
4. Why should result previews use projections and limits?

## Further reading

- PyMongo documentation: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/
- PyMongo collection API: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/
- MongoDB connection strings: https://www.mongodb.com/docs/manual/reference/connection-string/

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.